# Carte de circulation des illustrations d'Ovide

Reconstruction de la cellule perdue qui génère `resultats/Datavis/carte_circulation.html` :
une carte Leaflet animée (slider 1490-1750) montrant les villes d'édition des *Métamorphoses*
d'Ovide et les flèches de circulation d'un même graveur entre plusieurs villes.

Les données proviennent du tableau de référence **`retours_celine/BNU_corpus.ods`**
(feuille `Synthèse`), le corpus documenté par Céline. La logique des flèches (quel graveur
relie quelles villes) est déduite automatiquement de ce tableau.

In [44]:
import os
import re
import json
from odf.opendocument import load as charger_ods
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

RACINE = os.path.abspath("../..")
DOSSIER_VIZ = os.path.join(RACINE, "resultats", "Datavis")
os.makedirs(DOSSIER_VIZ, exist_ok=True)
CHEMIN_SORTIE = os.path.join(DOSSIER_VIZ, "carte_circulation.html")
CHEMIN_CORPUS = os.path.join(RACINE, "retours_celine", "BNU_corpus.ods")

## 1. Chargement du corpus (`BNU_corpus.ods`, feuille `Synthèse`)

Chaque ligne du tableau est une édition (titre, ville, année, technique, graveur, liens).
On lit la feuille cellule par cellule via `odfpy` plutôt qu'avec `pandas.read_excel` : ce
fichier a une colonne **`titre abrégé`** (juste avant `titre complet`) que le lecteur ODF de
pandas ignore silencieusement (avec deux autres colonnes annexes) — probablement des colonnes
masquées dans le tableur. C'est ce `titre abrégé` qui est utilisé pour l'affichage sur la carte.

Les noms de villes sont saisis en texte libre (variantes, noms latins, crochets...) : on les
fait correspondre à une liste de 19 villes canoniques avec leurs coordonnées, pour pouvoir les
placer sur la carte. Les éditions sans ville exploitable (`s.l.`, ville manquante) sont écartées.

In [45]:
# Ville canonique -> (latitude, longitude)
VILLES_COORDS = {
    "Amsterdam": (52.3676, 4.9041), "Anvers": (51.2194, 4.4025), "Arnhem": (51.9851, 5.8987),
    "Augsbourg": (48.3705, 10.8978), "Bruxelles": (50.8503, 4.3517), "Cologne": (50.9375, 6.9603),
    "Francfort": (50.1109, 8.6821), "Haarlem": (52.3874, 4.6462), "Leyde": (52.1601, 4.497),
    "Londres": (51.5074, -0.1278), "Lyon": (45.764, 4.8357), "Mayence": (49.9929, 8.2473),
    "Nuremberg": (49.4521, 11.0767), "Paris": (48.8566, 2.3522), "Rouen": (49.4432, 1.0993),
    "Toscolano": (45.65, 10.6167), "Valladolid": (41.6523, -4.7245), "Venise": (45.4408, 12.3155),
    "Vienne": (48.2082, 16.3738),
}

# Variante de nom de ville trouvée dans le tableau -> ville canonique
CORRESPONDANCE_VILLES = {
    "Amsterdam": "Amsterdam", "Anvers": "Anvers", "Anvers / Rotterdam": "Anvers",
    "Arnhemii": "Arnhem", "Augsbourg": "Augsbourg", "Bruxelles": "Bruxelles",
    "Cöllen": "Cologne", "[Köln]": "Cologne",
    "Francfort": "Francfort", "Francfort-sur-le-Main": "Francfort",
    "Leyde": "Leyde", "Londres": "Londres", "Lyon": "Lyon",
    "Mayence (Meinz)": "Mayence", "Nuremberg": "Nuremberg",
    "Paris": "Paris", "[Paris]": "Paris", "Rouen": "Rouen",
    "Tusculanum": "Toscolano", "Valladolid": "Valladolid", "Venise": "Venise",
    "Vienne": "Vienne", "Vienne\xa0?": "Vienne", "[Haarlem]": "Haarlem",
}

def lire_feuille_ods(chemin, nom_feuille):
    """Lit une feuille ODS cellule par cellule (voir note ci-dessus sur les limites de pandas)."""
    doc = charger_ods(chemin)
    table = next(t for t in doc.spreadsheet.getElementsByType(Table)
                 if t.getAttribute("name") == nom_feuille)
    lignes_brutes = table.getElementsByType(TableRow)

    def valeurs_ligne(ligne):
        valeurs, col = {}, 0
        for cellule in ligne.getElementsByType(TableCell):
            rep = cellule.getAttribute("numbercolumnsrepeated")
            rep = int(rep) if rep else 1
            paras = cellule.getElementsByType(P)
            texte = " ".join(teletype.extractText(p) for p in paras)
            for k in range(rep):
                valeurs[col + k] = texte
            col += rep
        return valeurs

    entetes = valeurs_ligne(lignes_brutes[0])
    colonnes = {i: t.strip() for i, t in entetes.items() if t.strip()}

    lignes = []
    for ligne in lignes_brutes[1:]:
        rep = ligne.getAttribute("numberrowsrepeated")
        rep = int(rep) if rep else 1
        valeurs = valeurs_ligne(ligne)
        if not any(v.strip() for v in valeurs.values()):
            continue  # ligne vide (fin de feuille)
        d = {nom: valeurs.get(i, "").strip() for i, nom in colonnes.items()}
        lignes.extend([d] * rep)
    return lignes

def extraire_annee(valeur):
    """Renvoie la première année à 4 chiffres trouvée (ex: "1527 / 1528 ?" -> 1527)."""
    m = re.search(r"\d{4}", str(valeur))
    return int(m.group()) if m else None

def extraire_lien(row):
    """Choisit le premier lien exploitable, par ordre de préférence."""
    for col in ["version numérisée 1", "version numérisée 2", "Biblioteca Digital Ovidiana", "url catalogue"]:
        val = row.get(col, "")
        if not val:
            continue
        premier = val.split(";")[0].strip()
        if premier.startswith("http"):
            return premier
    return None

def abreger_titre(titre, longueur=90):
    """Titre abrégé : coupe au dernier mot entier avant `longueur` caractères, avec '…'."""
    titre = str(titre).strip()
    if len(titre) <= longueur:
        return titre
    coupe = titre[:longueur].rsplit(" ", 1)[0]
    return coupe.rstrip(" ,;:.") + "…"

def normaliser_graveur(nom):
    """Retire les dates entre parenthèses et la ponctuation superflue (réutilisé pour les
    flèches de circulation et les flèches de copie)."""
    nom = re.sub(r"\(.*?\)", "", nom)
    nom = nom.strip().rstrip(",").strip()
    return re.sub(r"\s+", " ", nom)

In [46]:
COL_GRAVEUR = "graveur\xa0: Nom, Prénom"
corpus = lire_feuille_ods(CHEMIN_CORPUS, "Synthèse")

villes = {}
non_localisees = 0
for row in corpus:
    ville_brute = row.get("ville", "")
    if not ville_brute:
        non_localisees += 1
        continue
    ville = CORRESPONDANCE_VILLES.get(ville_brute)
    if ville is None:
        non_localisees += 1
        continue

    titre_source = row.get("titre abrégé") or row.get("titre complet") or ""
    edition = {
        "titre": abreger_titre(titre_source) if titre_source else "",
        "annee": row.get("année", ""),
        "an": extraire_annee(row.get("année", "")),
        "graveur": row.get(COL_GRAVEUR) or None,
        "technique": row.get("technique") or "?",
        "lien": extraire_lien(row),
        "copies": row.get("copies de cette édition", "").strip(),
    }
    lat, lon = VILLES_COORDS[ville]
    villes.setdefault(ville, {"lat": lat, "lon": lon, "editions": []})["editions"].append(edition)

villes = dict(sorted(villes.items()))

print(len(villes), "villes,", sum(len(d["editions"]) for d in villes.values()), "éditions")
print(non_localisees, "éditions écartées (ville non localisable)")

nb_anonymes = sum(
    1 for d in villes.values() for e in d["editions"]
    if e["graveur"] and str(e["graveur"]).strip().lower().startswith("anonyme")
)
print(nb_anonymes, "éditions à graveur anonyme, bien listées dans les popups des villes")

19 villes, 108 éditions
3 éditions écartées (ville non localisable)
36 éditions à graveur anonyme, bien listées dans les popups des villes


## 2. Flèches de circulation

Pour chaque graveur identifié, on repère la première année d'apparition dans chaque ville.
Les mentions "AnonymeXXXX" désignent en réalité un même graveur anonyme (identifié par son
année) : si "AnonymeXXXX" réapparaît dans une autre ville, on considère qu'il s'agit du même
graveur et on trace une flèche, comme pour un graveur nommé. Seules les mentions non
identifiantes ("?", "inaccessible") ou les entrées sans année sont écartées.
Si un graveur apparaît dans au moins deux villes distinctes, on relie ces villes par ordre
chronologique en une chaîne continue (ville la plus ancienne → suivante → suivante...),
plutôt que de faire repartir toutes les flèches de la même ville d'origine.

In [47]:
def graveur_identifiable(nom):
    """Un 'AnonymeXXXX' est un identifiant valable (même graveur anonyme si XXXX se répète) ;
    seules les mentions non identifiantes sont écartées."""
    if not nom:
        return False
    if nom.strip() in {"?", "inaccessible"}:
        return False
    return True

# graveur normalisé -> {ville: premiere_annee}
premiere_annee_par_ville = {}
for ville, d in villes.items():
    for e in d["editions"]:
        if e["an"] is None or not graveur_identifiable(e["graveur"]):
            continue
        g = normaliser_graveur(e["graveur"])
        villes_g = premiere_annee_par_ville.setdefault(g, {})
        if ville not in villes_g or e["an"] < villes_g[ville]:
            villes_g[ville] = e["an"]

# Palette cyclique (mêmes couleurs que l'original, dans l'ordre alphabétique des graveurs)
PALETTE = ["#c0392b", "#1565c0", "#2e7d32", "#8e44ad", "#e67e22",
           "#16a085", "#d35400", "#2980b9", "#7f8c8d", "#c2185b"]

graveurs_multi_villes = {
    g: v for g, v in premiere_annee_par_ville.items() if len(v) >= 2
}

couleurs = {
    g: PALETTE[i % len(PALETTE)]
    for i, g in enumerate(sorted(graveurs_multi_villes))
}

# Chaîne chronologique : ville1 -> ville2 -> ville3 ... (pas une étoile depuis la 1ère ville)
fleches = []
for g in sorted(graveurs_multi_villes):
    trajet = sorted(graveurs_multi_villes[g].items(), key=lambda kv: kv[1])
    for (origine, an_orig), (dest, an_dest) in zip(trajet, trajet[1:]):
        fleches.append({
            "graveur": g,
            "couleur": couleurs[g],
            "origine": origine,
            "dest": dest,
            "an_orig": an_orig,
            "an_dest": an_dest,
            "c1": [villes[origine]["lat"], villes[origine]["lon"]],
            "c2": [villes[dest]["lat"], villes[dest]["lon"]],
        })

print(len(fleches), "flèches pour", len(couleurs), "graveurs")
for f in fleches:
    print(f["graveur"], "-", f["origine"], "->", f["dest"], f"({f['an_dest']})")

7 flèches pour 6 graveurs
Baur, Johann Wilhelm - Vienne -> Augsbourg (1709)
Baur, Johann Wilhelm - Augsbourg -> Nuremberg (1718)
Leroy II, Guillaume - Lyon -> Venise (1516)
Passe, Crispin de - Cologne -> Arnhem (1607)
Salomon, Bernard - Lyon -> Paris (1570)
Solis, Virgil - Francfort -> Anvers (1595)
Tempesta, Antonio - Anvers -> Amsterdam (1610)


## 3. Flèches de copie (colonne `copies de cette édition`)

Cette colonne du tableau indique, en texte libre, de quel(s) graveur(s) les illustrations
d'une édition sont des copies (ex. `"copie Bernard Salomon"`, `"copie X et Y"`,
`"même famille que X"`). On en extrait un ou plusieurs noms de graveurs candidats, qu'on fait
correspondre au registre des graveurs connus (même logique de normalisation que pour les
flèches de circulation). Pour chaque correspondance trouvée, on trace une flèche **en
pointillés** depuis la plus récente édition connue de ce graveur antérieure ou égale à
l'édition copieuse, vers cette édition copieuse.

**Toute mention ambiguë est écartée sans flèche** : dès que le texte contient "ou" ou un "?"
(attribution hésitante entre plusieurs graveurs, ex. `"copie X ou Y"`, `"X\xa0? Y\xa0?"`), on
ne peut pas savoir laquelle est la bonne, donc on ne trace rien pour cette édition plutôt que
de deviner. Sont également écartées : les mentions non identifiantes ("QUOI ?"), les renvois
à d'autres lignes du tableau ("l'un des suivants"), les notes de travail ("reprendre ici"), et
les incohérences chronologiques (le graveur cité n'a aucune édition connue avant celle qui est
censée le copier). Le journal ci-dessous liste tout ce qui a été écarté et pourquoi.

In [48]:
def normaliser_candidat_anonyme(c):
    """« Anonyme 1563 » ou un simple « 1572 » -> 'AnonymeXXXX' (même convention que le registre)."""
    c = re.sub(r"anonyme\s*(\d{4})", r"Anonyme\1", c, flags=re.IGNORECASE)
    if re.fullmatch(r"\d{4}", c.strip()):
        c = "Anonyme" + c.strip()
    return c.strip()

def eclater_enumeration(fragment):
    """« Anonyme1563, 1572 » -> ['Anonyme1563', '1572'] si tout ressemble à une liste d'années."""
    parties = [p.strip() for p in fragment.split(",")]
    if len(parties) > 1 and all(re.fullmatch(r"(anonyme\s*)?\d{4}", p, re.IGNORECASE) for p in parties):
        return parties
    return [fragment]

def mention_ambigue(texte):
    """"ou" ou "?" signale une attribution hésitante entre plusieurs graveurs : on ne devine pas."""
    return bool(re.search(r"\bou\b", texte, re.IGNORECASE)) or "?" in texte

def extraire_candidats_copie(texte):
    """« copie X et Y » / « même famille que X, Y et Z » -> liste de noms de graveurs candidats.
    (Les mentions ambiguës "ou"/"?" ont déjà été écartées par mention_ambigue avant cet appel.)"""
    t = re.sub(r"\(.*?\)", "", texte)
    t = re.sub(r"^\s*(copie|même famille que)\s*", "", t, flags=re.IGNORECASE)
    bruts = re.split(r"\s+et\s+", t)
    candidats = []
    for c in bruts:
        c = c.strip(" .,;?\xa0")
        if not c:
            continue
        for sous in eclater_enumeration(c):
            sous = normaliser_candidat_anonyme(sous.strip(" .,;?\xa0"))
            if sous:
                candidats.append(sous)
    return candidats

def tokens_nom(nom):
    nom = re.sub(r"\(.*?\)", "", nom)
    nom = nom.replace(",", " ")
    return set(t.lower() for t in re.findall(r"[a-zà-öø-ÿ']+", nom) if len(t) >= 3)

def trouver_graveur_connu(candidat, registre_toutes_annees):
    """Fait correspondre un nom en texte libre à une clé du registre (chevauchement de mots)."""
    if candidat in registre_toutes_annees:
        return candidat
    tc = tokens_nom(candidat)
    if not tc:
        return None
    meilleur, meilleur_score = None, 0
    for nom_reg in registre_toutes_annees:
        if nom_reg.lower().startswith("anonyme"):
            continue  # déjà couvert par la correspondance exacte ci-dessus
        score = len(tc & tokens_nom(nom_reg))
        if score > meilleur_score:
            meilleur, meilleur_score = nom_reg, score
    return meilleur if meilleur_score >= 1 else None

# registre : graveur normalisé -> liste de TOUTES ses (ville, année) [pas seulement la 1ère par ville]
toutes_instances_par_graveur = {}
for ville, d in villes.items():
    for e in d["editions"]:
        if e["an"] is None or not e["graveur"]:
            continue
        g = normaliser_graveur(e["graveur"])
        toutes_instances_par_graveur.setdefault(g, []).append((ville, e["an"]))

fleches_copies = []
non_resolus = []
for ville, d in villes.items():
    for e in d["editions"]:
        if not e["copies"] or e["an"] is None:
            continue

        if mention_ambigue(e["copies"]):
            non_resolus.append((ville, e["an"], e["titre"], e["copies"], "(toute la mention)", "attribution ambiguë (ou/possibilité multiple)"))
            continue

        candidats = extraire_candidats_copie(e["copies"])
        matches_uniques = {}
        for c in candidats:
            match = trouver_graveur_connu(c, toutes_instances_par_graveur)
            if match is None:
                non_resolus.append((ville, e["an"], e["titre"], e["copies"], c, "aucune correspondance"))
            else:
                matches_uniques.setdefault(match, []).append(c)

        for match, candidats_source in matches_uniques.items():
            instances_anterieures = [
                inst for inst in toutes_instances_par_graveur[match]
                if inst[1] <= e["an"] and inst != (ville, e["an"])
            ]
            if not instances_anterieures:
                non_resolus.append((ville, e["an"], e["titre"], e["copies"], match, "pas d'édition antérieure de ce graveur"))
                continue
            origine, an_origine = max(instances_anterieures, key=lambda x: x[1])
            fleches_copies.append({
                "graveur": match,
                "origine": origine,
                "dest": ville,
                "an_orig": an_origine,
                "an_dest": e["an"],
                "titre_copie": e["titre"],
                "c1": [villes[origine]["lat"], villes[origine]["lon"]],
                "c2": [villes[ville]["lat"], villes[ville]["lon"]],
            })

# on nettoie : le champ "copies" n'était utile qu'ici, pas dans la carte finale
for d in villes.values():
    for e in d["editions"]:
        e.pop("copies", None)

print(len(fleches_copies), "flèches de copie résolues")
print(len(non_resolus), "mentions non résolues :")
for ville, an, titre, texte, cible, raison in non_resolus:
    print(f"  ✗ {ville} {an} « {titre[:50]} » — {texte!r} — {cible!r} ({raison})")

23 flèches de copie résolues
10 mentions non résolues :
  ✗ Amsterdam 1683 « P. Ovidii Nasonis opera omnia » — 'copie Clein, Francisco (inv.) et Savery, Salomon (sculp.) ou Philippe, Pierre' — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Amsterdam 1693 « Les Metamorphoses d'Ovide, avec des explications à » — "copie Clein, Francisco (inv.) et Savery, Salomon (sculp.) ou l'un des suivants" — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Cologne 1602 « Metamorphoseon Ovidianarum per Crispianum Passaeum » — 'copie van der Borcht\xa0? Salomon\xa0?' — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Lyon 1527 « Publii Ovidii Nasonis Sulmonensis Metamorphoseos L » — 'copie Leroy II, Guillaume ou 1497' — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Lyon 1532 « Le Grand Olympe » — 'copie Leroy II, Guillaume ou Anonyme1497 ou Anonyme1527' — '(toute la mention)' (attribution ambiguë (ou/poss

## 4. Génération de la carte

Fond de carte **OpenHistoricalMap** (frontières et toponymes réels d'époque, pas juste un
style graphique) au lieu d'OpenStreetMap : fond en tuiles vectorielles (MapLibre GL), intégré
à la carte Leaflet via le plugin `maplibre-gl-leaflet`, filtré par date avec le plugin
`maplibre-gl-dates` pour **suivre le slider** (les frontières/noms affichés correspondent à
l'année sélectionnée, 1490-1750).

Raffinements par rapport à la version précédente :
- **Flèches courbes** (plutôt que des segments droits) avec un léger halo blanc pour rester
  lisibles sur le fond de carte historique, désormais chargé, et une **zone de survol
  invisible plus large** pour afficher facilement l'infobulle au survol (les traits fins
  étaient difficiles à cibler avec la souris).
- Flèches de circulation : trait plein coloré. Flèches de copie : pointillés bruns, animés
  (léger effet de "fourmis en marche") pour bien les distinguer visuellement des flèches de
  circulation.
- Les marqueurs de ville sont désormais mis à jour en place (au lieu d'être détruits et
  recréés à chaque déplacement du slider) : leur taille s'anime en douceur (rebond léger en
  CSS) quand le nombre d'éditions cumulées change, au lieu de changer brutalement.
- Le texte "Point ∝ nb d'éditions..." était mal placé et peu clair : il est remplacé par une
  phrase en clair, intégrée à la légende plutôt que posée entre les boutons et la légende.

In [49]:
TEMPLATE_HTML = r"""<!DOCTYPE html>
<html lang="fr"><head><meta charset="utf-8">
<title>Circulation des illustrations d'Ovide</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet-polylinedecorator@1.6.0/dist/leaflet.polylineDecorator.js"></script>
<link href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css" rel="stylesheet"/>
<script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>
<script src="https://unpkg.com/@openhistoricalmap/maplibre-gl-dates@1.3.0/index.js"></script>
<script src="https://unpkg.com/@maplibre/maplibre-gl-leaflet@0.0.20/leaflet-maplibre-gl.js"></script>
<style>
  body { margin:0; font-family:Georgia,serif; }
  #map { height:100vh; }
  .panneau { position:absolute; top:12px; left:50px; z-index:1000; background:rgba(255,255,255,.94);
    padding:14px 18px; border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.3); max-width:320px; }
  .panneau h1 { font-size:16px; margin:0 0 6px; color:#5a3e2b; }
  .panneau p { font-size:12px; margin:3px 0; color:#555; }
  #annee { font-size:26px; font-weight:bold; color:#5a3e2b; }
  #slider { width:100%; margin-top:8px; }
  .legende { margin-top:10px; font-size:11px; }
  .legende .rubrique { font-weight:bold; color:#5a3e2b; margin-top:8px; }
  .legende .note { color:#777; font-style:italic; margin:3px 0; }
  .legende div.item { margin:2px 0; }
  .legende span.puce { display:inline-block; width:22px; height:3px; vertical-align:middle; margin-right:6px; }
  .legende span.pointille { border-top:2px dashed #6d4c41; background:none; height:0; }
  button { background:#5a3e2b; color:#fff; border:none; padding:6px 14px; border-radius:4px;
    cursor:pointer; font-family:Georgia,serif; margin-right:6px; }

  /* Infobulles : thème parchemin cohérent avec le panneau */
  .leaflet-tooltip { font-family:Georgia,serif; font-size:12px; background:#fffaf0;
    border:1px solid #5a3e2b; color:#3e2c23; padding:4px 9px; box-shadow:0 1px 5px rgba(0,0,0,.3); }
  .leaflet-tooltip-top:before { border-top-color:#5a3e2b; }
  .leaflet-tooltip-bottom:before { border-bottom-color:#5a3e2b; }

  .leaflet-popup-content { font-family:Georgia,serif; font-size:12px; max-height:280px; overflow-y:auto; }
  .leaflet-popup-content h3 { margin:0 0 6px; color:#5a3e2b; font-size:14px; }
  .ed { border-bottom:1px solid #eee; padding:4px 0; }
  .ed a { color:#1565c0; text-decoration:none; }

  /* Marqueurs de ville : rebond léger quand leur taille change */
  .marqueur-ville { transition: r .5s cubic-bezier(.34,1.56,.64,1); }

  /* Flèches (halo + trait) : fondu à l'apparition */
  .flux-anim { transition: opacity .5s ease-out; }
  /* Flèches de copie : pointillés "fourmis en marche" */
  .flux-copie { animation: pointille-anime 1s linear infinite; }
  @keyframes pointille-anime { to { stroke-dashoffset: -20; } }
  /* Flèches de circulation : léger relief façon gravure */
  .flux-circulation { filter: drop-shadow(0 0 1px rgba(0,0,0,.35)); }
</style></head><body>
<div id="map"></div>
<div class="panneau">
  <h1>Circulation des illustrations d'Ovide</h1>
  <p>Année : <span id="annee"></span></p>
  <input type="range" id="slider" min="1490" max="1750" value="1750" step="1">
  <p><button onclick="play()">▶ Animer</button><button onclick="pause()">⏸</button>
     <button onclick="tout()">Tout</button></p>
  <div class="legende" id="legende"></div>
</div>
<script>
  const villes = __VILLES__;
  const fleches = __FLECHES__;
  const couleurs = __COULEURS__;
  const flechesCopies = __FLECHES_COPIES__;

  // --- Légende (construite une seule fois : elle ne dépend pas de l'année) ---
  let lg = '<div class="rubrique">Villes</div>';
  lg += '<div class="note">Taille du point ∝ racine du nombre d\'éditions connues à cette date</div>';
  lg += '<div class="rubrique">Graveurs qui circulent</div>';
  for (const [g,c] of Object.entries(couleurs))
    lg += '<div class="item"><span class="puce" style="background:'+c+'"></span>'+g+'</div>';
  lg += '<div class="rubrique">Copies</div>';
  lg += '<div class="item"><span class="puce pointille"></span>édition copiant un autre graveur</div>';
  document.getElementById('legende').innerHTML = lg;

  const map = L.map('map').setView([48.5,7],5);

  // Fond de carte historique (OpenHistoricalMap, frontières/toponymes d'époque),
  // filtré par date pour suivre le slider.
  const fondHistorique = L.maplibreGL({
    style: 'https://www.openhistoricalmap.org/map-styles/main/main.json',
    attribution: '© OpenHistoricalMap contributors'
  }).addTo(map);
  const carteHistorique = fondHistorique.getMaplibreMap();

  function filtrerParDate(annee) {
    try {
      if (carteHistorique.isStyleLoaded && carteHistorique.isStyleLoaded() && carteHistorique.filterByDate) {
        carteHistorique.filterByDate(String(annee));
      }
    } catch (err) { /* style pas encore prêt, on réessaiera au prochain dessiner() */ }
  }
  carteHistorique.on('load', () => filtrerParDate(+document.getElementById('slider').value));

  // --- Petit arc entre deux points, pour des flèches courbes plutôt que des segments droits ---
  function pointsArc(c1, c2, courbure) {
    const [lat1, lon1] = c1, [lat2, lon2] = c2;
    const dx = lon2 - lon1, dy = lat2 - lat1;
    const longueur = Math.sqrt(dx*dx + dy*dy) || 1e-9;
    const px = dy / longueur, py = -dx / longueur;  // perpendiculaire unitaire
    const cx = (lon1 + lon2) / 2 + px * longueur * courbure;
    const cy = (lat1 + lat2) / 2 + py * longueur * courbure;
    const pts = [];
    const n = 24;
    for (let i = 0; i <= n; i++) {
      const t = i / n;
      const lat = (1-t)*(1-t)*lat1 + 2*(1-t)*t*cy + t*t*lat2;
      const lon = (1-t)*(1-t)*lon1 + 2*(1-t)*t*cx + t*t*lon2;
      pts.push([lat, lon]);
    }
    return pts;
  }

  // Trace une flèche courbe avec halo + zone de survol élargie + infobulle + pointe,
  // et la fait apparaître en fondu. options: {couleur, poids, pointille (bool), classe}
  function tracerFleche(c1, c2, infobulle, options) {
    const pts = pointsArc(c1, c2, 0.12);
    const calques = [];
    const cibles = [];  // opacité finale de chaque calque animable, pour le fondu d'entrée

    const opaciteHalo = .55;
    const halo = L.polyline(pts, {
      color:'#fff', weight: options.poids + 3, opacity: 0, interactive:false, className:'flux-anim'
    }).addTo(map);
    calques.push(halo); cibles.push(opaciteHalo);

    const opaciteTrait = .85;
    const styleTrait = {
      color: options.couleur, weight: options.poids, opacity: 0, interactive:false,
      className: 'flux-anim ' + options.classe
    };
    if (options.pointille) styleTrait.dashArray = '4,7';
    const trait = L.polyline(pts, styleTrait).addTo(map);
    calques.push(trait); cibles.push(opaciteTrait);

    const deco = L.polylineDecorator(trait, {
      patterns: [{
        offset:'60%', repeat:0,
        symbol: L.Symbol.arrowHead({
          pixelSize: options.pointille ? 9 : 12,
          pathOptions: {color: options.couleur, fillOpacity:.9, weight:0}
        })
      }]
    }).addTo(map);
    calques.push(deco); cibles.push(undefined);  // pas de fondu sur la pointe (calque composite)

    // Zone de survol invisible et large : plus facile à cibler qu'un trait fin.
    const zoneSurvol = L.polyline(pts, {opacity:0, weight:16}).addTo(map);
    zoneSurvol.bindTooltip(infobulle, {sticky:true});
    calques.push(zoneSurvol); cibles.push(undefined);  // reste invisible : pas de fondu à faire

    // Déclenche le fondu au prochain "tick" (sinon le navigateur ne voit pas la transition
    // depuis opacity:0, puisque l'élément vient d'être créé avec cette valeur).
    setTimeout(() => {
      calques.forEach((c, i) => { if (cibles[i] !== undefined) c.setStyle({opacity: cibles[i]}); });
    }, 20);

    return calques;
  }

  // --- Marqueurs de ville : mis à jour en place (pas détruits/recréés) pour permettre
  // une transition CSS douce sur leur taille (voir .marqueur-ville) ---
  let mVilles = {};
  // --- Flèches : ajoutées/retirées seulement quand nécessaire (pas à chaque frame),
  // pour permettre une apparition en fondu plutôt qu'un pop-in répété ---
  let mFlechesActives = {}, mFlechesCopiesActives = {};

  function majFlechesActives(liste, actives, annee, construire) {
    const voulues = new Set();
    liste.forEach((f, i) => {
      if (f.an_dest > annee) return;
      voulues.add(i);
      if (!actives[i]) actives[i] = construire(f);
    });
    for (const cle of Object.keys(actives)) {
      if (!voulues.has(+cle)) {
        actives[cle].forEach(c => map.removeLayer(c));
        delete actives[cle];
      }
    }
  }

  function dessiner(annee) {
    document.getElementById('annee').textContent = annee;
    filtrerParDate(annee);

    for (const [ville,d] of Object.entries(villes)) {
      const editionsVisibles = d.editions.filter(e => e.an !== null && e.an <= annee);
      const n = editionsVisibles.length;
      if (n === 0) {
        if (mVilles[ville]) { map.removeLayer(mVilles[ville]); delete mVilles[ville]; }
        continue;
      }
      const rayon = 6 + Math.sqrt(n) * 4;
      let html = '<h3>'+ville+' ('+n+' édition(s) jusqu\'en '+annee+')</h3>';
      editionsVisibles.slice().sort((a,b)=>b.an-a.an).forEach(e=>{
        html += '<div class="ed"><b>'+e.annee+'</b> — '+e.titre+
                '<br><i>'+e.graveur+' · '+e.technique+'</i>';
        if (e.lien) html += '<br><a href="'+e.lien+'" target="_blank">→ voir</a>';
        html += '</div>';
      });
      if (mVilles[ville]) {
        mVilles[ville].setRadius(rayon);
        mVilles[ville].setPopupContent(html);
      } else {
        mVilles[ville] = L.circleMarker([d.lat,d.lon], {
          radius: rayon, fillColor:'#5a3e2b', color:'#fff', weight:2, fillOpacity:.8,
          className: 'marqueur-ville'
        }).addTo(map).bindPopup(html, {maxWidth:320});
      }
    }

    majFlechesActives(fleches, mFlechesActives, annee, f => tracerFleche(
      f.c1, f.c2,
      f.graveur+' : '+f.origine+' ('+f.an_orig+') → '+f.dest+' ('+f.an_dest+')',
      {couleur: f.couleur, poids: 2.5, pointille: false, classe: 'flux-circulation'}
    ));

    majFlechesActives(flechesCopies, mFlechesCopiesActives, annee, f => tracerFleche(
      f.c1, f.c2,
      '« '+f.titre_copie+' » — '+f.dest+' ('+f.an_dest+') copie '+f.graveur+', '+f.origine+' ('+f.an_orig+')',
      {couleur: '#6d4c41', poids: 1.75, pointille: true, classe: 'flux-copie'}
    ));
  }

  const slider=document.getElementById('slider');
  slider.addEventListener('input',e=>dessiner(+e.target.value));
  let timer=null;
  function play() { pause(); let a=+slider.value>=1750?1490:+slider.value;
    timer=setInterval(()=>{ a+=2; if(a>1750){a=1750;pause();} slider.value=a; dessiner(a); },120); }
  function pause() { if(timer){clearInterval(timer);timer=null;} }
  function tout() { pause(); slider.value=1750; dessiner(1750); }
  dessiner(1750);
</script></body></html>"""

html_final = (TEMPLATE_HTML
    .replace("__VILLES__", json.dumps(villes, ensure_ascii=False))
    .replace("__FLECHES__", json.dumps(fleches, ensure_ascii=False))
    .replace("__COULEURS__", json.dumps(couleurs, ensure_ascii=False))
    .replace("__FLECHES_COPIES__", json.dumps(fleches_copies, ensure_ascii=False)))

with open(CHEMIN_SORTIE, "w", encoding="utf-8") as f:
    f.write(html_final)

print("Carte écrite dans", CHEMIN_SORTIE)

Carte écrite dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/Datavis/carte_circulation.html
